# Análisis de complejidad en señales EEG de sueño

**Dataset:** [Sleep-EDF Expanded (PhysioNet)](https://physionet.org/content/sleep-edfx/) - Kemp et al. 2000. Registros PSG de noche completa (EEG, EOG, EMG) con hipnogramas de etapas de sueño (W, N1, N2, N3, REM), uno de los datasets de referencia en la literatura de complejidad de EEG de sueño.

El objetivo es medir cómo cambia la complejidad de la señal EEG entre etapas de sueño, usando `mne` para la descarga y el preprocesamiento y `antropy` para las métricas de entropía / dimensión fractal.

## Flujo del análisis

La lógica reutilizable vive en el paquete [`src/`](./src); este notebook sólo la orquesta:

1. Descarga PSG + hipnograma de un sujeto/noche — `src.data.download_sleep_edf`.
2. Carga la señal y segmenta el EEG (canal Fpz-Cz) en épocas de 30 s etiquetadas por etapa de sueño (los estadios 3 y 4 se fusionan en N3) — `src.preprocessing`.
3. Calcula por época: entropía de permutación, entropía muestral, dimensión fractal de Higuchi y entropía espectral — `src.complexity`.
4. Promedia la complejidad por etapa (W, N1, N2, N3, REM). Típicamente se observa mayor complejidad en vigilia/REM y menor en sueño profundo (N3).
5. Exporta dos CSV a `data/processed/`.

In [ ]:
%pip install -q mne antropy pandas matplotlib

import sys
from pathlib import Path

# Permite importar el paquete `src` de la clase (la carpeta que contiene este notebook).
CLASE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
if str(CLASE_ROOT) not in sys.path:
    sys.path.insert(0, str(CLASE_ROOT))

import pandas as pd

from src.config import DATA_PROCESSED, EEG_CHANNEL
from src.data import download_sleep_edf
from src.preprocessing import load_recording, make_epochs
from src.complexity import complexity_dataframe, summarize_by_stage

## 1. Descarga del dataset

Un sujeto y una noche, para mantenerlo liviano (hay 83 sujetos disponibles). Los archivos se guardan en `data/raw/` (si ya existen no se vuelven a descargar).

In [2]:
psg_path, hypnogram_path = download_sleep_edf(subject=0, recording=1)
print(f"PSG:        {psg_path}")
print(f"Hipnograma: {hypnogram_path}")

PSG:        C:\workspace\IID_MP\data\raw\physionet-sleep-data\SC4001E0-PSG.edf
Hipnograma: C:\workspace\IID_MP\data\raw\physionet-sleep-data\SC4001EC-Hypnogram.edf


## 2. Carga de la señal y segmentación en épocas

Se conserva el canal `EEG Fpz-Cz` y se lo segmenta en épocas de 30 s etiquetadas
por etapa de sueño (los estadios 3 y 4 se fusionan en N3).

In [3]:
raw = load_recording(psg_path, hypnogram_path, channel=EEG_CHANNEL)
epochs = make_epochs(raw)

print(epochs)
print("\nÉpocas por etapa:")
print(pd.Series(epochs.events[:, -1]).map({v: k for k, v in epochs.event_id.items()}).value_counts())

c:\workspace\IID_MP\src\preprocessing.py:29: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
c:\workspace\IID_MP\src\preprocessing.py:29: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
c:\workspace\IID_MP\src\preprocessing.py:29: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)


<Epochs | 2650 events (all good), 0 – 29.99 s (baseline off), ~60.7 MiB, data loaded,
 'W': 1997
 'N1': 58
 'N2': 250
 'N3': 220
 'REM': 125>

Épocas por etapa:
W      1997
N2      250
N3      220
REM     125
N1       58
Name: count, dtype: int64


## 3. Métricas de complejidad por época

Por cada época de 30 s se calcula: entropía de permutación, entropía muestral,
dimensión fractal de Higuchi y entropía espectral.

In [4]:
df = complexity_dataframe(epochs)
df.head()

,etapa,perm_entropy,sample_entropy,higuchi_fd,spectral_entropy
0,W,0.975813,0.551161,1.428769,0.454394
1,W,0.992163,0.953349,1.677663,0.594885
2,W,0.998227,0.863745,1.681944,0.447458
3,W,0.985182,1.145231,1.565277,0.590497
4,W,0.973685,1.215904,1.515665,0.616270


## 4. Resumen: complejidad promedio por etapa

Típicamente se observa mayor complejidad en vigilia/REM y menor en sueño profundo (N3).

In [5]:
resumen = summarize_by_stage(df)
resumen

,perm_entropy,sample_entropy,higuchi_fd,spectral_entropy
etapa,,,,
W,0.988170,0.925981,1.596066,0.536449
N1,0.928542,1.075132,1.516409,0.634246
N2,0.873815,0.723310,1.367073,0.553069
N3,0.802642,0.498753,1.229428,0.458118
REM,0.927302,1.042598,1.475237,0.620675


In [6]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
por_epoca_csv = DATA_PROCESSED / "eeg_sleep_complexity_por_epoca.csv"
resumen_csv = DATA_PROCESSED / "eeg_sleep_complexity_resumen.csv"

df.to_csv(por_epoca_csv, index=False)
resumen.to_csv(resumen_csv)
print(f"Archivos guardados:\n  {por_epoca_csv}\n  {resumen_csv}")

Archivos guardados:
  C:\workspace\IID_MP\data\processed\eeg_sleep_complexity_por_epoca.csv
  C:\workspace\IID_MP\data\processed\eeg_sleep_complexity_resumen.csv
